In [1]:
import cv2
import numpy as np
from collections import deque

In [2]:
video_path = r"D:\Duy Toan\Python\DUT AI Club\Homework\day 28 3-3-2026\dataset.mp4"

n = 30
threshold_val = 30

In [3]:
def background_subtraction(video_path, n=10, threshold_val=30):
    cap = cv2.VideoCapture(video_path)
    frame_queue = deque(maxlen=n)
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frame_blur = cv2.GaussianBlur(frame, (5, 5), 0)
        frame_queue.append(frame_blur)
        frame_array = np.array(frame_queue)
        background = np.median(frame_array, axis=0).astype(np.uint8)

        diff = cv2.absdiff(frame, background)
        diff_gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
        _, mask = cv2.threshold(diff_gray, threshold_val, 255, cv2.THRESH_BINARY)
        kernel = np.ones((5, 5), np.uint8)
        mask = cv2.dilate(mask, kernel, iterations=2)
        mask = cv2.erode(mask, kernel, iterations=2)

        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for contour in contours:
            if cv2.contourArea(contour) < 500:
                continue
            (x, y, w, h) = cv2.boundingRect(contour)
            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.imshow("Median Object Detection", frame)
        cv2.imshow('Background', mask)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    cap.release()
    cv2.destroyAllWindows()

background_subtraction(video_path, n, threshold_val)    